# Joint geographic and temporal validation of WASH household models

Place this notebook beside **Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz** and select **Run → Run All Cells** in JupyterLab. You can also leave the household file in **climate_stack_outputs**. Keep it compressed.

If the imports fail, run **%pip install scikit-learn xgboost pandas numpy scipy matplotlib pillow python-docx** in a code cell, restart the kernel, and run all cells again. For Word 97–2003 DOC export, install LibreOffice Writer. The notebook always saves a DOCX first. If automatic conversion is unavailable, use **Save As → Word 97–2003 Document (*.doc)** in desktop Word.

The notebook uses the same full-resolution Nigeria ADM1 GeoJSON as the hotspot and Bayesian notebooks. It downloads the boundary if needed. If that fails, save [this boundary file](https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson) beside the notebook as **geoBoundaries-NGA-ADM1.geojson**.

Outputs go to **wash_joint_validation_outputs**. No additional climate files or raster grids are required for this validation step.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = Path.cwd()
INPUT = ROOT / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
if not INPUT.is_file():
    INPUT = ROOT / "climate_stack_outputs" / INPUT.name
if not INPUT.is_file():
    raise FileNotFoundError("Put the compressed climate-linked household file beside the notebook.")
OUTPUT = ROOT / "wash_joint_validation_outputs"
OUTPUT.mkdir(exist_ok=True)
KEY = ["survey_round","hv001","hv002"]
TARGET = "wash_dep_2plus"
CLIMATE = ["rain_mm_3m","temp_mean_c_3m"]
DEMOGRAPHIC = ["rural","head_female","head_age_years_capped","usual_household_size","nonwash_asset_count"]
FEATURE_SETS = {"Household only": DEMOGRAPHIC, "Household and climate": DEMOGRAPHIC + CLIMATE}
NAMES = dict(zip(DEMOGRAPHIC + CLIMATE, ["Rural residence","Female household head","Age of household head",
             "Household size","Non-WASH asset count","Rainfall over 3 months","Mean temperature over 3 months"]))
RAW = ["head_age_years_capped","usual_household_size","nonwash_asset_count",*CLIMATE]
columns = [*KEY,"state","zone","residence","head_sex","hh_weight","psu_key","stratum_hv023_key","wash_domains_known",TARGET,*RAW]
d = pd.read_csv(INPUT,usecols=columns,low_memory=False)
if d.groupby("survey_round").size().to_dict() != {2013:38522,2018:40427,2024:40047}:
    raise ValueError("Unexpected survey counts. Select the combined household output from the stack notebook.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household keys.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not d[TARGET].dropna().isin([0,1]).all() or d.loc[d.survey_round.eq(2013),TARGET].notna().any():
    raise ValueError("The file does not use the verified main 2018–2024 target.")
main_rounds = d.survey_round.isin([2018,2024])
if not d.loc[main_rounds,TARGET].notna().eq(d.loc[main_rounds,"wash_domains_known"].eq(3)).all():
    raise ValueError("The target does not match the complete-case eligibility rule.")
if d[["state","zone"]].isna().any().any() or d.groupby("state").zone.nunique().ne(1).any():
    raise ValueError("Each state needs a consistent zone label.")
for column in RAW:
    d[column] = pd.to_numeric(d[column],errors="raise")
    if not np.isfinite(d[column].dropna()).all():
        raise ValueError(f"Non-finite predictor values in {column}.")
if not d.residence.dropna().isin(["Rural","Urban"]).all() or not d.head_sex.dropna().isin(["Female","Male"]).all():
    raise ValueError("Unexpected residence or head-sex labels.")
d["rural"] = d.residence.map({"Rural":1.,"Urban":0.})
d["head_female"] = d.head_sex.map({"Female":1.,"Male":0.})
analysis = d.loc[main_rounds & d[TARGET].notna()].copy()
analysis[TARGET] = analysis[TARGET].astype(int)
if analysis.groupby("survey_round").size().to_dict() != {2018:36821,2024:36412}:
    raise ValueError("Unexpected complete-case totals. Review the target and household input.")
audit = []
for year,g in d.groupby("survey_round"):
    known = g[TARGET].notna()
    audit.append({
        "survey_round":year, "households":len(g), "known_target":int(known.sum()),
        "missing_target":int((~known).sum()), "weighted_target_known_pct":100*g.loc[known,"hh_weight"].sum()/g.hh_weight.sum(),
        "rainfall_missing_among_known":int(g.loc[known,CLIMATE[0]].isna().sum()),
        "temperature_missing_among_known":int(g.loc[known,CLIMATE[1]].isna().sum()),
    })
audit = pd.DataFrame(audit)
audit.to_csv(OUTPUT/"Household_Model_Input_QC.csv",index=False)
display(audit)


## 1. Hold out both geography and time

For each of the six geopolitical zones, train on **2018 households from the other five zones**, then predict **2024 households from the held-out zone**. Each eligible 2024 household receives one prediction per candidate from a model that has seen neither its zone nor any 2024 households. The test uses observed 2024 predictors and known outcomes. The 2013 round remains in the master data and eligibility audit.

Reuse the four previous candidates and their fixed settings. Medians and scaling are estimated only from the corresponding training fold. Pooled and zone-specific metrics use original 2024 household weights. Positive prediction bias means the model overestimates deprivation. The state map uses the prespecified logistic model with household and climate predictors for a consistent illustration.


In [ ]:
ZONES = ["North Central","North East","North West","South East","South South","South West"]
if set(analysis.zone) != set(ZONES):
    raise ValueError("The six geopolitical zones are required for the geographic evaluation.")

def fit_predict(train,test,features,learner):
    medians = train[features].median()
    if medians.isna().any():
        raise ValueError(f"A training feature is entirely missing: {medians[medians.isna()].index.tolist()}")
    xtrain = train[features].fillna(medians)
    xtest = test[features].fillna(medians)
    weight = train.hh_weight / train.hh_weight.mean()
    if learner == "Logistic":
        scaler = StandardScaler().fit(xtrain)
        model = LogisticRegression(C=1,max_iter=1000)
        model.fit(scaler.transform(xtrain),train[TARGET],sample_weight=weight)
        predicted = model.predict_proba(scaler.transform(xtest))[:,1]
    else:
        model = XGBClassifier(n_estimators=180,max_depth=3,learning_rate=.05,
                              min_child_weight=12,subsample=.9,colsample_bytree=.9,
                              tree_method="hist",random_state=20241111,n_jobs=2,
                              objective="binary:logistic",eval_metric="logloss")
        model.fit(xtrain,train[TARGET],sample_weight=weight)
        predicted = model.predict_proba(xtest)[:,1]
    return predicted,model,xtest

def summarize(test,predicted):
    y,w = test[TARGET],test.hh_weight
    if y.nunique() != 2:
        raise ValueError("The holdout must contain both outcome classes.")
    if not np.isfinite(predicted).all() or not ((predicted>=0)&(predicted<=1)).all():
        raise ValueError("Invalid predicted probabilities.")
    return {
        "n_test":len(test),"observed_prevalence":np.average(y,weights=w),
        "mean_prediction":np.average(predicted,weights=w),
        "auc":roc_auc_score(y,predicted,sample_weight=w),
        "average_precision":average_precision_score(y,predicted,sample_weight=w),
        "brier":brier_score_loss(y,predicted,sample_weight=w),
        "log_loss":log_loss(y,predicted,sample_weight=w,labels=[0,1]),
    }


train2018 = analysis.loc[analysis.survey_round.eq(2018)]
test2024 = analysis.loc[analysis.survey_round.eq(2024)].copy()
if set(train2018.zone) != set(ZONES) or set(test2024.zone) != set(ZONES):
    raise ValueError("Every zone must appear in both modelling rounds.")

candidates = {
    "Logistic household": ("Logistic","Household only"),
    "Logistic climate": ("Logistic","Household and climate"),
    "Boosted household": ("Boosted trees","Household only"),
    "Boosted climate": ("Boosted trees","Household and climate"),
}
predictions = pd.DataFrame(index=test2024.index,columns=list(candidates),dtype=float)
splits, zone_rows = [], []
for zone in ZONES:
    train = train2018.loc[train2018.zone.ne(zone)]
    test = test2024.loc[test2024.zone.eq(zone)]
    if set(train.state) & set(test.state) or set(train.index) & set(test.index):
        raise ValueError(f"Training and test data overlap for {zone}.")
    splits.append({"held_out_zone":zone,"train_round":2018,"test_round":2024,
                   "n_train":len(train),"n_test":len(test),"n_test_states":test.state.nunique()})
    for name,(learner,feature_set) in candidates.items():
        pred,_,_ = fit_predict(train,test,FEATURE_SETS[feature_set],learner)
        if predictions.loc[test.index,name].notna().any():
            raise ValueError("A test household appears in two validation folds.")
        predictions.loc[test.index,name] = pred
        stats = summarize(test,pred)
        zone_rows.append({"zone":zone,"model":name,**stats,
                          "bias_pp":100*(stats["mean_prediction"]-stats["observed_prevalence"])})
if predictions.isna().any().any():
    raise ValueError("Some 2024 households did not receive all four predictions.")
split_qc = pd.DataFrame(splits)
zone_results = pd.DataFrame(zone_rows)
pooled = pd.DataFrame([{"model":name,**summarize(test2024,predictions[name].to_numpy())} for name in candidates])
pooled["bias_pp"] = 100*(pooled.mean_prediction-pooled.observed_prevalence)
display(pooled.round(3))


## 2. Quantify uncertainty in prediction error

Resample the **2024 primary sampling units (PSUs) within survey strata**. Draw one fewer PSU than the number originally sampled in each stratum, with replacement, and rescale the replicate weights. The notebook uses the full 2024 PSU roster, including PSUs with no known outcome. It uses the same 999 replicates for all models and contrasts.

The percentile intervals describe test-sample variation conditional on the six fitted models and on households with known outcomes. They omit training uncertainty, missing-outcome bias, and finite population corrections. Differences use paired replicates. A negative Brier difference favours the first model named in the contrast. State errors are weighted predicted minus observed percentage points. State intervals are pointwise and exploratory. They do not support a formal hotspot designation.

References include [blocked cross-validation](https://doi.org/10.1111/ecog.02881) and [the Rao–Wu rescaled bootstrap](https://dennisfeehan.org/surveybootstrap/articles/rescaled_bootstrap.html).


In [ ]:
PSU, STRATUM = "psu_key","stratum_hv023_key"
full2024 = d.loc[d.survey_round.eq(2024),[PSU,STRATUM]]
if full2024.isna().any().any():
    raise ValueError("2024 design identifiers contain missing values.")
design = full2024.drop_duplicates().sort_values([STRATUM,PSU]).reset_index(drop=True)
if design[PSU].duplicated().any():
    raise ValueError("A 2024 PSU belongs to more than one stratum.")
if design.groupby(STRATUM).size().lt(2).any():
    raise ValueError("This bootstrap requires at least two PSUs per 2024 stratum.")

B = 999
rng = np.random.default_rng(20241112)
rep = np.zeros((B,len(design)))
for _,members in design.groupby(STRATUM,sort=False):
    positions = members.index.to_numpy()
    n = len(positions)
    rep[:,positions] = rng.multinomial(n-1,np.repeat(1/n,n),size=B)*n/(n-1)

psu_code = test2024[PSU].map(pd.Series(design.index,index=design[PSU]))
if psu_code.isna().any():
    raise ValueError("Could not find a test household's PSU.")
codes = psu_code.to_numpy(dtype=int)
w = test2024.hh_weight.to_numpy()
y = test2024[TARGET].to_numpy()

def totals(values, group_codes=codes):
    return np.bincount(group_codes,weights=values,minlength=len(design))

total_weights = totals(w)
rep_total = rep @ total_weights
if (rep_total<=0).any():
    raise ValueError("A replicate has no eligible test households.")

loss_reps = {}
for name in candidates:
    error = predictions[name].to_numpy()-y
    boot_brier = (rep @ totals(w*error**2))/rep_total
    boot_bias = 100*(rep @ totals(w*error))/rep_total
    loss_reps[name] = boot_brier
    row = pooled.model.eq(name)
    pooled.loc[row,["brier_ci_low","brier_ci_high"]] = np.quantile(boot_brier,[.025,.975])
    pooled.loc[row,["bias_ci_low_pp","bias_ci_high_pp"]] = np.quantile(boot_bias,[.025,.975])

pairs = [
    ("Logistic climate","Logistic household"),
    ("Boosted climate","Boosted household"),
    ("Boosted household","Logistic household"),
    ("Boosted climate","Logistic climate"),
]
brier = pooled.set_index("model").brier
contrasts = []
for first,second in pairs:
    lo,hi = np.quantile(loss_reps[first]-loss_reps[second],[.025,.975])
    contrasts.append({"model_1":first,"model_2":second,"delta_brier":brier[first]-brier[second],
                      "ci95_low":lo,"ci95_high":hi})
contrasts = pd.DataFrame(contrasts)

MAP_MODEL = "Logistic climate"
mapped_prediction = predictions[MAP_MODEL].to_numpy()
state_rows = []
for state in sorted(test2024.state.unique()):
    mask = test2024.state.eq(state).to_numpy()
    ww, yy, pp = w[mask], y[mask], mapped_prediction[mask]
    denominator = rep @ totals(ww,codes[mask])
    valid = denominator>0
    state_error = np.full(B,np.nan)
    state_error[valid] = 100*(rep[valid] @ totals(ww*(pp-yy),codes[mask]))/denominator[valid]
    low,high = np.nanquantile(state_error,[.025,.975]) if valid.sum()>=.95*B else (np.nan,np.nan)
    state_rows.append({"state":state,"model":MAP_MODEL,"n_known":int(mask.sum()),
                       "observed_pct":100*np.average(yy,weights=ww),
                       "predicted_pct":100*np.average(pp,weights=ww),
                       "bias_pp":100*np.average(pp-yy,weights=ww),
                       "ci95_low_pp":low,"ci95_high_pp":high,"valid_replicates":int(valid.sum())})
state_errors = pd.DataFrame(state_rows)
assert len(pooled)==4 and len(zone_results)==24 and len(state_errors)==37
assert split_qc.n_test.sum()==36412
assert np.isfinite(pooled[["brier_ci_low","brier_ci_high","bias_ci_low_pp","bias_ci_high_pp"]]).all().all()
for name,table in {
    "Joint_Validation_Metrics.csv":pooled,
    "Joint_Validation_Zone_Metrics.csv":zone_results,
    "Joint_Validation_Splits.csv":split_qc,
    "Joint_Validation_Paired_Brier.csv":contrasts,
    "Joint_Validation_State_Errors.csv":state_errors,
}.items():
    table.to_csv(OUTPUT/name,index=False)
display(pooled.round(3))
display(contrasts.round(4))


## 3. Export the report and 600 dpi JPEGs

The zone chart compares all four candidates. The state map shows 2024 prediction errors for the prespecified logistic model with climate. Hatching marks state intervals that include zero or are unavailable. The map depicts errors among **sampled households with known outcomes**, using the observed predictors in their holdout. It does not extend prediction to unsampled areas.


In [ ]:
from urllib.request import urlopen
import hashlib
import json
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Patch,PathPatch
from matplotlib.path import Path as PlotPath
BOUNDARY = ROOT/"geoBoundaries-NGA-ADM1.geojson"
URL = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/"
       "releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1.geojson")
CREDIT = "GRID3 Nigeria ADM1 via geoBoundaries gbOpen, 2022 boundaries, CC BY 4.0"
if not BOUNDARY.is_file():
    try:
        with urlopen(URL,timeout=45) as response:
            raw = response.read()
        json.loads(raw)
    except (OSError,ValueError) as error:
        raise RuntimeError(f"Save the boundary file from the manual link beside the notebook as {BOUNDARY.name}.") from error
    BOUNDARY.write_bytes(raw)
raw = BOUNDARY.read_bytes()
digest = hashlib.sha256(raw).hexdigest()
(OUTPUT/"Boundary_Source.txt").write_text(f"{CREDIT}\n{URL}\nSHA-256 {digest}\n",encoding="utf-8")
features = {}
for feature in json.loads(raw)["features"]:
    name = feature["properties"]["shapeName"]
    state = "FCT" if name == "Abuja Federal Capital Territory" else name
    if state in features:
        raise ValueError(f"Duplicate boundary for {state}.")
    features[state] = feature["geometry"]
if set(features) != set(state_errors.state):
    raise ValueError("The boundary state names do not match the model results.")
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path



def polygon_path(geometry):
    kind = geometry["type"]
    if kind == "Polygon":
        polygons = [geometry["coordinates"]]
    elif kind == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry type: {kind}")
    paths = []
    for polygon in polygons:
        for i, ring in enumerate(polygon):
            points = np.asarray(ring, dtype=float)[:, :2]
            if len(points) < 4 or not np.isfinite(points).all():
                raise ValueError("Malformed boundary ring.")
            if not np.array_equal(points[0], points[-1]):
                points = np.vstack([points, points[0]])
            x, y = points.T
            signed_area = np.sum(x[:-1] * y[1:] - y[:-1] * x[1:])
            # Opposite winding keeps interior holes open.
            if (signed_area > 0) != (i == 0):
                points = points[::-1]
            codes = np.full(len(points), PlotPath.LINETO, dtype=np.uint8)
            codes[0], codes[-1] = PlotPath.MOVETO, PlotPath.CLOSEPOLY
            paths.append(PlotPath(points, codes))
    return PlotPath.make_compound_path(*paths)

paths = {state: polygon_path(geometry) for state, geometry in features.items()}
coordinates = np.vstack([path.vertices for path in paths.values()])
west, south = coordinates.min(axis=0)
east, north = coordinates.max(axis=0)
if not (0 < west < east < 20 and 0 < south < north < 20):
    raise ValueError("Unexpected boundary extent. Use Nigeria ADM1 longitude and latitude.")
aspect = 1 / np.cos(np.deg2rad((south + north) / 2))


colors = ["#1E6785","#A45129","#5A4D92","#356944"]
fig,ax = plt.subplots(figsize=(7,3.5),layout="constrained")
for (i,(model,group)),color in zip(enumerate(zone_results.groupby("model",sort=False)),colors):
    group = group.set_index("zone").loc[ZONES]
    ax.scatter(np.arange(len(ZONES))+(i-1.5)*.14,group.bias_pp,label=model,color=color,s=24)
ax.axhline(0,linestyle="--",color="#777777",linewidth=.9)
ax.set(xticks=np.arange(len(ZONES)),xticklabels=ZONES,ylabel="Predicted minus observed (pp)",
       title="2024 prediction bias by held-out zone")
ax.legend(fontsize=8,ncol=2,frameon=False)
ax.grid(axis="y",color="#DFE4E8")
zone_fig = save_figure(fig,"Figure_1_Joint_Validation_Zone_Bias_600dpi.jpg")

fig,ax = plt.subplots(figsize=(7,4.05))
fig.subplots_adjust(top=.90,bottom=.28)
fig.suptitle("2024 state prediction errors",fontsize=13,fontweight="bold",y=.97)
limit = max(5,5*np.ceil(state_errors.bias_pp.abs().max()/5))
norm,cmap = TwoSlopeNorm(0,-limit,limit),plt.get_cmap("RdBu_r")
state_lookup = state_errors.set_index("state")
for state,path in paths.items():
    r = state_lookup.loc[state]
    uncertain = pd.isna(r.ci95_low_pp) or r.ci95_low_pp<=0<=r.ci95_high_pp
    ax.add_patch(PathPatch(path,facecolor=cmap(norm(r.bias_pp)),edgecolor="#434A52",
                           linewidth=.35,hatch="////" if uncertain else None))
ax.set(xlim=(west-.2,east+.2),ylim=(south-.18,north+.18),aspect=aspect)
ax.set_axis_off()
bar=fig.colorbar(plt.cm.ScalarMappable(norm=norm,cmap=cmap),cax=fig.add_axes([.22,.22,.56,.028]),
                 orientation="horizontal")
bar.ax.tick_params(labelsize=8)
bar.set_label("Predicted minus observed (percentage points)",fontsize=8)
fig.legend(handles=[Patch(facecolor="white",edgecolor="#434A52",hatch="////",
                           label="Interval includes zero or is unavailable")],loc="lower center",
           bbox_to_anchor=(.5,.05),fontsize=8,frameon=False)
fig.text(.5,.025,CREDIT,ha="center",fontsize=6.5)
map_fig = save_figure(fig,"Figure_2_Joint_Validation_State_Bias_600dpi.jpg")
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Joint validation of household WASH models in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")



doc.add_paragraph("Joint validation of household WASH models in Nigeria",style="Title")
doc.add_paragraph("2018 training households and 2024 test households")
doc.add_paragraph(
    "Each fold holds out one geopolitical zone from 2018 training and evaluates predictions in that zone in 2024. "
    "The four candidate models use the same 36,412 test households with known deprivation outcomes."
)
doc.add_heading("Weighted predictive performance",level=1)
rows = [[r.model,f"{r.auc:.3f}",f"{r.brier:.3f}",f"{r.brier_ci_low:.3f} to {r.brier_ci_high:.3f}",
         f"{r.bias_pp:+.1f}",f"{r.bias_ci_low_pp:+.1f} to {r.bias_ci_high_pp:+.1f}"]
        for r in pooled.itertuples(index=False)]
add_table("Table 1. Pooled performance and conditional 95% intervals",
          ["Model","AUC","Brier","Brier interval","Bias (pp)","Bias interval (pp)"],
          rows,[1.40,.60,.70,1.35,.95,2.00],{0})
rows = [[r.model_1,r.model_2,f"{r.delta_brier:+.4f}",f"{r.ci95_low:+.4f} to {r.ci95_high:+.4f}"]
        for r in contrasts.itertuples(index=False)]
add_table("Table 2. Paired Brier comparisons",
          ["First model","Second model","Difference","95% interval"],
          rows,[1.5,1.5,1.25,2.75],{0,1})
doc.add_paragraph(
    "Lower Brier scores indicate better probability predictions. Negative paired differences favour the first model. "
    "Positive prediction bias indicates overestimation. Intervals use 999 rescaled PSU bootstrap replicates "
    "within 2024 survey strata with fitted models held fixed."
)
doc.add_heading("Geographic pattern of prediction errors",level=1).paragraph_format.page_break_before=True
add_figure(zone_fig,"Figure 1. Weighted prediction bias for all four candidates.",
           "Dot plot of signed prediction bias by geopolitical zone.")
add_figure(map_fig,"Figure 2. State bias for logistic regression with household and climate predictors.",
           "State map of predicted minus observed deprivation among eligible 2024 households.")
doc.add_heading("State prediction errors",level=1).paragraph_format.page_break_before=True
rows = [[r.state,f"{r.n_known:,}",f"{r.observed_pct:.1f}",f"{r.predicted_pct:.1f}",f"{r.bias_pp:+.1f}",
         f"{r.ci95_low_pp:+.1f} to {r.ci95_high_pp:+.1f}" if pd.notna(r.ci95_low_pp) else "—"]
        for r in state_errors.itertuples(index=False)]
add_table("Table 3. Prespecified logistic model with climate in sampled households",
          ["State or FCT","Known n","Observed %","Predicted %","Bias (pp)","95% interval (pp)"],
          rows,[1.3,.8,1.1,1.1,1.0,1.7],{0},font_size=8.5)
doc.add_paragraph("State intervals are pointwise. These results apply to sampled households with known outcomes.",style="Caption")
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Joint_Validation_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, CSVs and two verified 600 dpi JPEGs in {OUTPUT.resolve()}.")
